In [1]:
def extract_spans(prediction, text=None):
    """
    Extract valid and invalid spans from NER predictions following BIOES tagging scheme.
    
    Args:
        prediction (list): List of dictionaries, each containing a token and its BIOES tag
        text (str, optional): Original text input (not used in this implementation but included for future extensions)
    
    Returns:
        dict: Dictionary with two keys:
            - 'valid_spans': List of valid entity spans
            - 'invalid_spans': List of invalid entity spans
    """
    valid_spans = []
    invalid_spans = []
    
    current_span = []
    current_entity_type = None
    
    # Helper function to check if a tag follows BIOES scheme
    def is_valid_bioes_transition(prev_tag, current_tag):
        if prev_tag is None:
            return current_tag.startswith('B-') or current_tag.startswith('S-')
        
        prev_prefix = prev_tag[0]
        prev_entity = prev_tag[2:] if len(prev_tag) > 2 else ''
        current_prefix = current_tag[0]
        current_entity = current_tag[2:] if len(current_tag) > 2 else ''
        
        # Entity types must match within a span
        if prev_entity != current_entity and prev_prefix in ['B', 'I'] and current_prefix in ['I', 'E']:
            return False
            
        # Valid transitions
        if prev_prefix == 'B':
            return (current_prefix == 'I' or current_prefix == 'E')
        elif prev_prefix == 'I':
            return (current_prefix == 'I' or current_prefix == 'E')
        elif prev_prefix == 'E' or prev_prefix == 'S':
            return (current_prefix == 'B' or current_prefix == 'S')
        
        return False
    
    prev_tag = None
    for token_dict in prediction:
        token = list(token_dict.keys())[0]
        tag = token_dict[token]
        
        # Process the tag
        if tag.startswith('B-'):  # Beginning of a span
            # If we were building a span, finalize it
            if current_span:
                # Check if the previous span ended properly (with E- or S-)
                if not (prev_tag.startswith('E-') or prev_tag.startswith('S-')):
                    invalid_spans.append((current_span, current_entity_type))
                else:
                    valid_spans.append((current_span, current_entity_type))
                current_span = []
                
            current_span.append(token)
            current_entity_type = tag[2:]  # Extract entity type (after 'B-')
        
        elif tag.startswith('I-'):  # Inside of a span
            # Must follow B- or I- of same entity type
            if (prev_tag and (prev_tag.startswith('B-') or prev_tag.startswith('I-')) and 
                prev_tag[2:] == tag[2:]):
                current_span.append(token)
            else:
                # Invalid: I- tag not following proper B-/I- tag
                if current_span:
                    invalid_spans.append((current_span, current_entity_type))
                current_span = [token]
                current_entity_type = tag[2:]
        
        elif tag.startswith('E-'):  # End of a span
            # Must follow B- or I- of same entity type
            if (prev_tag and (prev_tag.startswith('B-') or prev_tag.startswith('I-')) and 
                prev_tag[2:] == tag[2:]):
                current_span.append(token)
                valid_spans.append((current_span, current_entity_type))
                current_span = []
                current_entity_type = None
            else:
                # Invalid: E- tag not following proper B-/I- tag
                if current_span:
                    invalid_spans.append((current_span, current_entity_type))
                invalid_spans.append(([token], tag[2:]))
                current_span = []
                current_entity_type = None
        
        elif tag.startswith('S-'):  # Single token span
            # If we were building a span, finalize it (as invalid)
            if current_span:
                invalid_spans.append((current_span, current_entity_type))
                current_span = []
            
            # Add this as a single-token valid span
            valid_spans.append(([token], tag[2:]))
        
        elif tag == 'O':  # Outside any span
            # If we were building a span, finalize it (as invalid)
            if current_span:
                invalid_spans.append((current_span, current_entity_type))
                current_span = []
                current_entity_type = None
        
        else:
            # Unknown tag format
            if current_span:
                invalid_spans.append((current_span, current_entity_type))
                current_span = []
            invalid_spans.append(([token], "Unknown"))
        
        prev_tag = tag
    
    # Handle any remaining span
    if current_span:
        # Check if it's a valid span (must end with E-)
        if prev_tag and prev_tag.startswith('E-'):
            valid_spans.append((current_span, current_entity_type))
        else:
            invalid_spans.append((current_span, current_entity_type))
    
    # Format the results
    formatted_valid_spans = []
    for span_tokens, entity_type in valid_spans:
        formatted_valid_spans.append({
            'text': ' '.join(span_tokens),
            'tokens': span_tokens,
            'entity_type': entity_type
        })
    
    formatted_invalid_spans = []
    for span_tokens, entity_type in invalid_spans:
        formatted_invalid_spans.append({
            'text': ' '.join(span_tokens),
            'tokens': span_tokens,
            'entity_type': entity_type,
        })
    
    return {
        'valid_spans': formatted_valid_spans,
        'invalid_spans': formatted_invalid_spans
    }


In [2]:
import os

def export_predictions_to_conll(predictions_output, output_filepath):
    """
    Exports the predictions from simpletransformers' .predict() method
    to a CoNLL formatted file.

    The CoNLL format expects one token per line, followed by its entity tag,
    with an empty line separating sentences.

    Args:
        predictions_output (list): A list of lists of dictionaries,
                                   where each inner list represents a sentence
                                   and each dictionary contains 'token' and 'entity' keys.
                                   Example: [[{'token': 'Barack', 'entity': 'B-PER'},
                                              {'token': 'Obama', 'entity': 'I-PER'}],
                                             [{'token': 'lives', 'entity': 'O'},
                                              {'token': 'in', 'entity': 'O'},
                                              {'token': 'Washington', 'entity': 'B-LOC'},
                                              {'token': 'D.C.', 'entity': 'I-LOC'}]]
        output_filepath (str): The path to the file where the CoNLL output will be saved.
    """
    try:
        with open(output_filepath, 'w', encoding='utf-8') as f:
            for sentence_predictions in predictions_output:
                for token_data in sentence_predictions:
                    # Ensure both 'token' and 'entity' keys exist
                    token = token_data.get('token', '')
                    entity = token_data.get('entity', 'O') # Default to 'O' if entity missing
                    f.write(f"{token}\t{entity}\n")
                f.write("\n")  # Add an empty line after each sentence
        print(f"✅ Predictions successfully exported to: {os.path.abspath(output_filepath)}")
    except Exception as e:
        print(f"❌ An error occurred while exporting predictions: {e}")

# --- Example Usage ---
# if __name__ == "__main__":
#     # This is a mock output from model.predict() for demonstration purposes.
#     # In your actual code, this would come directly from your simpletransformers model.
#     mock_predictions = [
#         [
#             {'token': 'Apple', 'entity': 'B-ORG'},
#             {'token': 'Inc.', 'entity': 'I-ORG'},
#             {'token': 'is', 'entity': 'O'},
#             {'token': 'headquartered', 'entity': 'O'},
#             {'token': 'in', 'entity': 'O'},
#             {'token': 'Cupertino', 'entity': 'B-LOC'},
#             {'token': ',', 'entity': 'O'},
#             {'token': 'California', 'entity': 'I-LOC'},
#             {'token': '.', 'entity': 'O'}
#         ],
#         [
#             {'token': 'Dr.', 'entity': 'O'},
#             {'token': 'Smith', 'entity': 'B-PER'},
#             {'token': 'visited', 'entity': 'O'},
#             {'token': 'Paris', 'entity': 'B-LOC'},
#             {'token': 'last', 'entity': 'O'},
#             {'token': 'summer', 'entity': 'O'},
#             {'token': '.' , 'entity': 'O'}
#         ]
#     ]

#     output_file = "predictions.conll"
#     export_predictions_to_conll(mock_predictions, output_file)

#     # You can also test with an empty list or a list with missing keys
#     print("\n--- Testing with empty predictions ---")
#     export_predictions_to_conll([], "empty_predictions.conll")

#     print("\n--- Testing with partially formed predictions ---")
#     partially_formed_predictions = [
#         [{'token': 'OnlyToken'}, {'entity': 'B-ORG'}], # Missing token/entity
#         [{'token': 'Google', 'entity': 'B-ORG'}]
#     ]
#     export_predictions_to_conll(partially_formed_predictions, "partially_formed.conll")

In [5]:
import os
import torch
from simpletransformers.ner import NERModel
labels = ['O',
                'B-Event', 'I-Event', 'E-Event', 'S-Event',
                'B-NonEvent', 'I-NonEvent', 'E-NonEvent', 'S-NonEvent',
                ]
# --- Configuration ---
# Replace 'your_username/your_model_repo' with your actual Hugging Face model repository ID.
# Example: 'bert-base-cased-finetuned-ner'
# You can also specify a local path if the model is downloaded.
HUGGING_FACE_MODEL_PATH = "ADFLER-xlnet-base-cased" # ⬅️ IMPORTANT: Update this!
MODEL_TYPE = "xlnet" # Or "roberta", "xlmroberta", etc., depending on your trained model
MODEL_NAME = "xlnet-base-cased" # Or "roberta-base", "xlm-roberta-base", etc.
                             # This should match the base model you used for training.

# --- Your Preprocessing Function (Placeholder) ---
# You mentioned you have a script to preprocess raw text into a tokenized list of tokens.
# Please replace this placeholder function with your actual preprocessing logic.
# This function should take raw text (str) and return a list of lists of tokens (List[List[str]])
# Each inner list represents a sentence, and contains the tokens for that sentence.
# Example: "Hello world. How are you?" -> [["Hello", "world", "."], ["How", "are", "you", "?"]]
import re
from typing import List

def tokenize_raw_text(text: str) -> List[str]:
    """
    Tokenizes a single string into a list of tokens based on spaces
    and punctuation, correctly handling decimals, units, and complex units.
    
    Args:
        text: The input string to tokenize.
    
    Returns:
        A list of tokens.
    """
    # This regex is updated to include '/' in the list of standalone punctuation.
    tokens = re.findall(r"\d+(?:\.\d+)?(?:[a-zA-Z\/°]+)?|\d+(?:\.\d+)?|[\w']+|[.,!?;:()\[\]\/-]+", text)
    return tokens

def tokenize_test_set(raw_messages: List[str]) -> List[List[str]]:
    """
    Tokenizes a list of raw messages (e.g., a test set).
    
    Args:
        raw_messages: A list of raw message strings.
        
    Returns:
        A list of lists of tokens.
    """
    tokenized_messages = []
    for message in raw_messages:
        tokenized_messages.append(tokenize_raw_text(message))
    return tokenized_messages


# --- Main Prediction Logic ---
def perform_ner_prediction(raw_text_input: str):
    """
    Loads an NER model from Hugging Face and performs predictions on new text data.

    Args:
        raw_text_input (str): The raw text on which to perform NER.
    """
    print(f"Attempting to load model from: {HUGGING_FACE_MODEL_PATH}")

    try:
        # Load the NER model from Hugging Face.
        # simpletransformers automatically handles downloading and loading.
        # Ensure you specify the correct model_type and model_name that were used during training.
        model = NERModel(
            model_type=MODEL_TYPE,
            model_name=HUGGING_FACE_MODEL_PATH, # Pass the Hugging Face repo ID here
            labels=labels,
            use_cuda=True if torch.cuda.is_available() else False # Use GPU if available
        )
        print("Model loaded successfully!")

        # Preprocess the input raw text using your custom script
        # The 'predict' method expects a list of lists of tokens.
        # Each inner list is a sentence, tokenized.
        preprocessed_sentences = tokenize_test_set(raw_text_input)

        if not preprocessed_sentences:
            print("Preprocessing resulted in no valid sentences. Cannot perform prediction.")
            return []

        print(f"Preprocessed input for prediction: {preprocessed_sentences}")

        # Perform predictions
        # The result will be a list of lists of dictionaries.
        # Each inner list corresponds to a sentence.
        # Each dictionary contains 'token' and 'entity_group' (or 'label' for older versions).
        predictions, raw_outputs = model.predict(preprocessed_sentences, split_on_space=False)
        # return predictions
        result = extract_spans(predictions[0])
        
        print("Valid spans:")
        for span in result['valid_spans']:
            print(f"- {span['text']} ({span['entity_type']})")
        
        print("\nInvalid spans:")
        for span in result['invalid_spans']:
            print(f"- {span['text']} ({span['entity_type']})")
        # print("\n--- Predictions ---")
        # for i, sentence_predictions in enumerate(result):
        #     print(f"Sentence {i+1}:")
        #     for token_info in sentence_predictions:
        #         print(f"  Token: '{token_info['word']}' -> Entity: '{token_info.get('entity', 'O')}'") # 'entity' or 'entity_group'

        # Optional: Print raw outputs if needed for debugging/analysis
        # print("\n--- Raw Outputs ---")
        # print(raw_outputs)

        return predictions

    except Exception as e:
        print(f"An error occurred: {e}")
        print("Please ensure:")
        print(f"1. '{HUGGING_FACE_MODEL_PATH}' is the correct Hugging Face repository ID for your model.")
        print(f"2. Your internet connection is stable if downloading for the first time.")
        print(f"3. The 'MODEL_TYPE' ('{MODEL_TYPE}') and 'MODEL_NAME' ('{MODEL_NAME}') match what you used during training.")
        print(f"4. Your 'preprocess_text' function returns the expected format (list of lists of strings).")
        return []

In [6]:
import pandas as pd
import os

vto_data = pd.read_csv(os.path.join('..', '..', 'processed', 'vto_labs', 'parsed-cleansed.csv'))
# ⬅️ IMPORTANT: Remember to update HUGGING_FACE_MODEL_PATH above!
predictions = perform_ner_prediction(vto_data['message'].to_list())
export_predictions_to_conll(predictions, 'prediction.conll')


Attempting to load model from: ADFLER-xlnet-base-cased
Model loaded successfully!
Preprocessed input for prediction: [['Taking', 'Off', '.'], ['Taking', 'Off', '.'], ['Taking', 'Off', '.'], ['Taking', 'Off', '.'], ['Home', 'Point', 'Recorded', '.', 'RTH', 'Altitude', ':', '30m', '.'], ['Home', 'Point', 'Recorded', '.', 'RTH', 'Altitude', ':', '30m', '.'], ['Camera', 'Settings', 'Adjusted', 'to', 'ActiveTrack'], ['Your', 'palm', 'is', 'too', 'far', 'away', 'from', 'the', 'aircraft', '.', 'Please', 'step', 'closer', '.'], ['Your', 'palm', 'is', 'too', 'far', 'away', 'from', 'the', 'aircraft', '.', 'Please', 'step', 'closer', '.'], ['Your', 'palm', 'is', 'too', 'close', 'to', 'the', 'aircraft', '.', 'Please', 'step', 'farther', 'away', '.'], ['PalmControl', 'in', 'Process'], ['PalmControl', 'in', 'Process'], ['Your', 'palm', 'is', 'too', 'close', 'to', 'the', 'aircraft', '.', 'Please', 'step', 'farther', 'away', '.'], ['Your', 'palm', 'is', 'too', 'far', 'away', 'from', 'the', 'aircraft',

Running Prediction: 100%|██████████| 19/19 [07:24<00:00, 23.39s/it]


Valid spans:
- Taking Off (Event)

Invalid spans:
✅ Predictions successfully exported to: d:\Data Kuliah\Semester 8\IEEE - TIFS\dataset\annotated\event_recognition\prediction.conll
Attempting to load model from: ADFLER-xlnet-base-cased
Model loaded successfully!
Preprocessed input for prediction: [['Google', 'was', 'founded', 'by', 'Larry', 'Page', 'and', 'Sergey', 'Brin', '.']]


Running Prediction: 100%|██████████| 1/1 [00:00<00:00,  1.98it/s]

Valid spans:
- Google was founded by Larry Page and Sergey Brin (Event)

Invalid spans:


[[{'Google': 'B-Event'},
  {'was': 'I-Event'},
  {'founded': 'I-Event'},
  {'by': 'I-Event'},
  {'Larry': 'I-Event'},
  {'Page': 'I-Event'},
  {'and': 'I-Event'},
  {'Sergey': 'I-Event'},
  {'Brin': 'E-Event'},
  {'.': 'O'}]]

In [7]:
predictions

[[{'Taking': 'B-Event'}, {'Off': 'E-Event'}, {'.': 'O'}],
 [{'Taking': 'B-Event'}, {'Off': 'E-Event'}, {'.': 'O'}],
 [{'Taking': 'B-Event'}, {'Off': 'E-Event'}, {'.': 'O'}],
 [{'Taking': 'B-Event'}, {'Off': 'E-Event'}, {'.': 'O'}],
 [{'Home': 'B-Event'},
  {'Point': 'I-Event'},
  {'Recorded': 'E-Event'},
  {'.': 'O'},
  {'RTH': 'B-NonEvent'},
  {'Altitude': 'I-NonEvent'},
  {':': 'I-NonEvent'},
  {'30m': 'E-NonEvent'},
  {'.': 'O'}],
 [{'Home': 'B-Event'},
  {'Point': 'I-Event'},
  {'Recorded': 'E-Event'},
  {'.': 'O'},
  {'RTH': 'B-NonEvent'},
  {'Altitude': 'I-NonEvent'},
  {':': 'I-NonEvent'},
  {'30m': 'E-NonEvent'},
  {'.': 'O'}],
 [{'Camera': 'B-Event'},
  {'Settings': 'I-Event'},
  {'Adjusted': 'I-Event'},
  {'to': 'I-Event'},
  {'ActiveTrack': 'E-Event'}],
 [{'Your': 'B-Event'},
  {'palm': 'I-Event'},
  {'is': 'I-Event'},
  {'too': 'I-Event'},
  {'far': 'I-Event'},
  {'away': 'I-Event'},
  {'from': 'I-Event'},
  {'the': 'I-Event'},
  {'aircraft': 'E-Event'},
  {'.': 'O'},
  {'P